# 04 — Feature Engineering

Este notebook construye las matrices de features `X` para los tres grafos del dataset Meetup Tennessee:
- **M** — Grafo de miembros
- **G** — Grafo de grupos
- **MG** — Grafo bipartito miembro-grupo

El output son tres objetos `torch_geometric.data.Data` guardados en disco, listos para los notebooks del GAE.

## Contexto y objetivo

Un Graph Autoencoder (GAE) requiere dos inputs: la **matriz de adyacencia** `A`, que codifica la estructura del grafo, y la **matriz de features** `X`, que describe los atributos de cada nodo. Este notebook se ocupa de construir `X` para cada grafo a partir de los metadatos disponibles.

A diferencia de otros enfoques que incluyen métricas estructurales derivadas del propio grafo (degree, betweenness, clustering...) como features, aquí se opta por usar **únicamente atributos de los metadatos originales**. Esta decisión es deliberada: las métricas estructurales contienen información que el GAE ya va a aprender por sí solo a partir de `A`, por lo que incluirlas en `X` introduciría redundancia y podría sesgar el modelo hacia patrones que ya conoce.

Dado que los modelos de ML y DL solo operan con valores numéricos, todos los atributos categóricos o textuales deben transformarse antes de construir `X`. Ese proceso de transformación es el **feature engineering** que se documenta a continuación.

### Decisiones de feature engineering

| Grafo | Nodo | Features |
|-------|------|----------|
| M | miembro | `location_level`, `lat`, `lon` |
| G | grupo | `log_num_members`, `log_num_events`, `is_truncated`, `has_valid_organizer`, one-hot `category_name` |
| MG | miembro | mismas que M (0 para features de grupo) |
| MG | grupo | mismas que G (0 para features de miembro) |

## 0. Imports y configuración

In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

DATA_PATH = '../data/processed/'
OUTPUT_PATH = '../data/graph_data/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

c:\Users\marco\.conda\envs\TFM_grafos_V2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga de datos

In [2]:
# Metadatos
meta_members = pd.read_csv(os.path.join(DATA_PATH, 'meta-members.csv'))
meta_groups  = pd.read_csv(os.path.join(DATA_PATH, 'meta-groups.csv'))
meta_events  = pd.read_csv(os.path.join(DATA_PATH, 'meta-events.csv'))

# Aristas
member_edges       = pd.read_csv(os.path.join(DATA_PATH, 'member-edges.csv'))
group_edges        = pd.read_csv(os.path.join(DATA_PATH, 'group-edges.csv'))
member_group_edges = pd.read_csv(os.path.join(DATA_PATH, 'member-to-group-edges.csv'))

print(f'meta_members      : {meta_members.shape}')
print(f'meta_groups       : {meta_groups.shape}')
print(f'meta_events       : {meta_events.shape}')
print(f'member_edges      : {member_edges.shape}')
print(f'group_edges       : {group_edges.shape}')
print(f'member_group_edges: {member_group_edges.shape}')

meta_members      : (24590, 6)
meta_groups       : (602, 7)
meta_events       : (19307, 4)
member_edges      : (1176368, 3)
group_edges       : (6692, 3)
member_group_edges: (45583, 3)


## 2. Feature Engineering — Nodos miembro

Los metadatos de miembro (`meta_members`) contienen seis columnas: `member_id`, `name`, `city`, `state`, `lat` y `lon`. De estas, `name` no aporta información modelable y `city` y `state` son variables categóricas de alta cardinalidad (893 ciudades distintas, 64 valores de estado) que no pueden usarse directamente.

### `location_level` — variable ordinal geográfica

En lugar de codificar `city` y `state` por separado con técnicas como one-hot encoding (inviable para 893 ciudades) o target encoding, se construye una **única variable ordinal** que captura la jerarquía geográfica natural del dataset:

| Valor | Significado |
|-------|-------------|
| 0 | Internacional — `state` es NaN (usuarios fuera de EE.UU.) |
| 1 | USA fuera de Tennessee — `state` válido pero distinto de TN, o corrupto |
| 2 | Tennessee fuera de Nashville — `state == 'TN'` y `city != 'Nashville'` |
| 3 | Nashville — `state == 'TN'` y `city == 'Nashville'` |

Esta codificación tiene varias ventajas frente a las alternativas. Primero, evita la explosión de dimensionalidad del one-hot. Segundo, captura la proximidad geográfica de forma implícita — valores más altos indican mayor cercanía al núcleo del dataset. Tercero, maneja de forma natural los dos problemas de calidad identificados en el EDA: los 118 miembros internacionales (level 0) y los 20 miembros con códigos de estado inválidos, que se asignan a level 1 al no poder confirmarse su ubicación exacta dentro de EE.UU.

### `lat` y `lon` — coordenadas geográficas

Se mantienen como features numéricas continuas. Aportan información geográfica más granular que `location_level` — dos miembros de Tennessee fuera de Nashville (ambos level 2) pueden estar en extremos opuestos del estado. 

**Limitación conocida:** el 28.9% de los miembros tiene asignadas las coordenadas del centroide de su ciudad en lugar de su ubicación real, cuando el usuario no especifica dirección concreta. Esto introduce ruido a nivel individual pero no invalida la feature — los centroides siguen siendo geográficamente coherentes a nivel de ciudad.

In [3]:
# Estados válidos de EE.UU.
VALID_US_STATES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA',
    'KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT',
    'VA','WA','WV','WI','WY','DC'
}

def compute_location_level(row):
    """
    0 — Internacional: state es NaN (usuarios fuera de EE.UU./Canadá)
    1 — USA fuera de TN: state válido pero no TN, o state inválido (corrupto)
    2 — Tennessee fuera de Nashville: state == 'TN' y city != 'Nashville'
    3 — Nashville: state == 'TN' y city == 'Nashville'
    """
    state = row['state']
    city  = row['city']

    if pd.isna(state):
        return 0  # Internacional
    if state not in VALID_US_STATES:
        return 1  # Estado corrupto — tratado como USA desconocido
    if state != 'TN':
        return 1  # USA fuera de TN
    if str(city).strip().lower() != 'nashville':
        return 2  # Tennessee fuera de Nashville
    return 3      # Nashville

meta_members['location_level'] = meta_members.apply(compute_location_level, axis=1)

print('Distribución de location_level:')
print(meta_members['location_level'].value_counts().sort_index())
print(f'\nTotal miembros: {len(meta_members)}')

Distribución de location_level:
location_level
0      118
1     1912
2     7781
3    14779
Name: count, dtype: int64

Total miembros: 24590


In [4]:
# Seleccionamos las tres features finales e indexamos por member_id
# city y state se descartan — su información queda capturada en location_level
member_features = meta_members[['member_id', 'location_level', 'lat', 'lon']].copy()
member_features = member_features.set_index('member_id')

print('Features de miembro:')
print(member_features.head())
print(f'\nShape: {member_features.shape}')
print(f'Nulos:\n{member_features.isnull().sum()}')

Features de miembro:
           location_level    lat    lon
member_id                              
2069                    2  36.00 -86.79
8386                    3  36.07 -86.78
9205                    2  36.00 -86.79
17903                   3  36.13 -86.80
20418                   3  36.17 -86.72

Shape: (24590, 3)
Nulos:
location_level    0
lat               0
lon               0
dtype: int64


## 3. Feature Engineering — Nodos grupo

Los metadatos de grupo (`meta_groups`) son más ricos que los de miembro, con siete columnas: `group_id`, `group_name`, `num_members`, `category_id`, `category_name`, `organizer_id` y `group_urlname`. Adicionalmente, `meta_events` permite agregar información de actividad por grupo.

### `log_num_members` — tamaño del grupo en escala logarítmica

El número de miembros registrados presenta una **distribución fuertemente asimétrica** (media 551, mediana 201, máximo 15.838). Aplicar el logaritmo — concretamente `log(num_members + 1)`, donde el `+1` evita `log(0)` para grupos con un único miembro — comprime la escala y reduce el efecto de los grupos gigantes como Nashville Hiking Meetup sobre el entrenamiento del modelo. Sin esta transformación, la diferencia bruta entre un grupo de 100 miembros y uno de 15.000 dominaría el espacio de features.

### `log_num_events` — actividad del grupo en escala logarítmica

`meta_events` no tiene una columna directa de número de eventos por grupo — hay que agregarlo contando las filas por `group_id`. Esta feature captura el nivel de actividad real del grupo, independientemente de su tamaño. Se aplica la misma transformación logarítmica por la misma razón de asimetría.

### `is_truncated` — indicador de truncamiento de la API

Identificado en el EDA, la API de Meetup devuelve como máximo 200 eventos por grupo. Los 32 grupos con exactamente 200 eventos registrados tienen en realidad más actividad de la que reflejan los datos — su `log_num_events` está artificialmente acotado. Esta variable binaria permite al modelo distinguir estos casos y no penalizar su reconstrucción por tener un valor de actividad que no es real.

### `has_valid_organizer` — integridad referencial del organizador

El EDA identificó 35 grupos cuyos `organizer_id` no tienen correspondencia en `meta_members`. Esto indica que el organizador es una entidad o cuenta corporativa no registrada como miembro individual. Esta variable binaria codifica esa anomalía de integridad referencial como una feature explícita.

### One-hot de `category_name` — categoría temática del grupo

`category_name` tiene 31 valores únicos — un número manejable para one-hot encoding. A diferencia de `city` en los miembros, aquí no existe una jerarquía natural entre categorías (Tech no es "más" que Dancing en ningún sentido ordinal), por lo que el one-hot es la codificación apropiada. Cada columna resultante es una variable binaria que indica si el grupo pertenece a esa categoría.

In [5]:
# Agregamos número de eventos por grupo desde meta_events
# meta_events tiene una fila por evento — contamos filas por group_id
events_per_group = meta_events.groupby('group_id').size().reset_index(name='num_events')
meta_groups = meta_groups.merge(events_per_group, on='group_id', how='left')
meta_groups['num_events'] = meta_groups['num_events'].fillna(0).astype(int)

print(f'Grupos con 0 eventos registrados: {(meta_groups["num_events"] == 0).sum()}')
print(f'Grupos con exactamente 200 eventos (truncados): {(meta_groups["num_events"] == 200).sum()}')

Grupos con 0 eventos registrados: 0
Grupos con exactamente 200 eventos (truncados): 32


In [6]:
# Transformación logarítmica: log1p = log(x + 1) para evitar log(0)
meta_groups['log_num_members'] = np.log1p(meta_groups['num_members'])
meta_groups['log_num_events']  = np.log1p(meta_groups['num_events'])

# Indicador de truncamiento: grupos con exactamente 200 eventos tienen actividad real desconocida
meta_groups['is_truncated'] = (meta_groups['num_events'] == 200).astype(int)

# Integridad referencial del organizador
valid_member_ids = set(meta_members['member_id'])
meta_groups['has_valid_organizer'] = meta_groups['organizer_id'].isin(valid_member_ids).astype(int)

print(f'Grupos con organizador válido: {meta_groups["has_valid_organizer"].sum()} / {len(meta_groups)}')
print(f'Grupos truncados             : {meta_groups["is_truncated"].sum()} / {len(meta_groups)}')

Grupos con organizador válido: 567 / 602
Grupos truncados             : 32 / 602


In [7]:
# One-hot encoding de category_name
# pd.get_dummies genera columnas booleanas en pandas recientes — se convierten a float más adelante
category_dummies = pd.get_dummies(meta_groups['category_name'], prefix='cat')

# Concatenamos todas las features de grupo
group_features = pd.concat([
    meta_groups[['group_id', 'log_num_members', 'log_num_events', 'is_truncated', 'has_valid_organizer']],
    category_dummies
], axis=1)
group_features = group_features.set_index('group_id')

print('Features de grupo:')
print(group_features.head())
print(f'\nShape: {group_features.shape}')
print(f'Nulos: {group_features.isnull().sum().sum()}')

Features de grupo:
          log_num_members  log_num_events  is_truncated  has_valid_organizer  \
group_id                                                                       
339011           9.670231        5.303305             1                    1   
19728145         7.483807        5.303305             1                    1   
6335372          7.962067        5.303305             1                    1   
10016242         7.588830        3.178054             0                    1   
21174496         7.931285        3.713572             0                    1   

          cat_Arts & Culture  cat_Book Clubs  cat_Career & Business  \
group_id                                                              
339011                 False           False                  False   
19728145               False           False                  False   
6335372                False           False                  False   
10016242               False           False                  Fal

## 4. Normalización

Antes de construir los tensores para PyTorch Geometric, se normalizan las features continuas con `StandardScaler` (media 0, desviación típica 1).

**¿Por qué normalizar?** Las redes neuronales son sensibles a la escala de los inputs. Sin normalización, features con rangos muy distintos (por ejemplo, `lat` en torno a 36 vs. `location_level` entre 0 y 3) provocarían que el gradiente estuviese dominado por las features de mayor magnitud, dificultando el entrenamiento.

**¿Qué se normaliza y qué no?**
- Features continuas (`location_level`, `lat`, `lon`, `log_num_members`, `log_num_events`): sí se normalizan. Aunque `location_level` es ordinal, su rango de 0-3 es distinto al de `lat` y `lon`, por lo que conviene escalarla.
- Features binarias (`is_truncated`, `has_valid_organizer`, columnas `cat_*`): no se normalizan. Ya están en el rango [0, 1] y escalarlas no aportaría nada.

**Importante:** el scaler se ajusta (`fit`) sobre el conjunto completo de metadatos, no sobre un subconjunto de entrenamiento. En detección de anomalías no supervisada no existe split train/test en el sentido tradicional — el modelo se entrena sobre todos los nodos y luego se puntúa cada uno.

In [8]:
MEMBER_TO_SCALE  = ['location_level', 'lat', 'lon']
GROUP_CONTINUOUS = ['log_num_members', 'log_num_events']

# Scaler para miembros
scaler_members = StandardScaler()
member_features_scaled = member_features.copy()
member_features_scaled[MEMBER_TO_SCALE] = scaler_members.fit_transform(
    member_features[MEMBER_TO_SCALE]
)

# Scaler para grupos
scaler_groups = StandardScaler()
group_features_scaled = group_features.copy()
group_features_scaled[GROUP_CONTINUOUS] = scaler_groups.fit_transform(
    group_features[GROUP_CONTINUOUS]
)

print('Member features escaladas:')
print(member_features_scaled.describe())
print('\nGroup features escaladas (continuas):')
print(group_features_scaled[GROUP_CONTINUOUS].describe())

Member features escaladas:
       location_level           lat           lon
count    2.459000e+04  2.459000e+04  2.459000e+04
mean    -1.571920e-16  1.712931e-15 -1.963745e-15
std      1.000020e+00  1.000020e+00  1.000020e+00
min     -3.815087e+00 -3.386782e+01 -7.002446e+00
25%     -7.796082e-01 -4.630394e-02 -3.516518e-02
50%      7.381311e-01 -2.572994e-03 -3.223035e-02
75%      7.381311e-01  1.800100e-03 -2.636069e-02
max      7.381311e-01  1.256133e+01  2.555555e+01

Group features escaladas (continuas):
       log_num_members  log_num_events
count     6.020000e+02    6.020000e+02
mean     -2.360607e-16    3.776971e-16
std       1.000832e+00    1.000832e+00
min      -2.877639e+00   -1.402717e+00
25%      -7.594637e-01   -8.895414e-01
50%      -1.885470e-02   -1.691633e-02
75%       7.194739e-01    6.441106e-01
max       2.939984e+00    2.010443e+00


## 5. Construcción del objeto PyG — Grafo M

PyTorch Geometric representa cada grafo como un objeto `Data` con tres componentes principales:
- `x`: matriz de features de forma `[num_nodes, num_features]`
- `edge_index`: tensor de forma `[2, num_edges]` con los índices de origen y destino de cada arista
- `edge_weight`: tensor de forma `[num_edges]` con el peso de cada arista

**Filtrado de nodos aislados:** `meta_members` contiene 24.590 miembros, pero `member-edges.csv` solo incluye pares de miembros con co-membresía — los miembros sin ninguna arista no aparecen en el archivo de aristas. El grafo M tiene por tanto 11.371 nodos, no 24.590. Incluir los nodos aislados en el objeto `Data` sería técnicamente posible pero contraproducente: el GAE no puede aprender ninguna representación útil de un nodo sin conexiones, ya que su encoder GCN opera precisamente sobre la estructura del grafo.

**Grafo no dirigido:** aunque `member-edges.csv` almacena cada arista una sola vez, PyTorch Geometric requiere que los grafos no dirigidos incluyan ambas direcciones en `edge_index`. Por eso se concatenan `[src, dst]` y `[dst, src]` — el número de aristas en el objeto Data es el doble que en el CSV original.

**Mapeo de IDs a índices:** PyG requiere que los nodos estén indexados con enteros consecutivos desde 0. Se construye un diccionario `member_id_to_idx` para mapear los `member_id` originales a estos índices, y se guarda en el objeto `Data` para poder interpretar los resultados del modelo.

In [9]:
# Solo miembros que aparecen en member_edges — descartamos nodos aislados
members_in_edges = set(member_edges['member1']).union(set(member_edges['member2']))
member_features_M = member_features_scaled[
    member_features_scaled.index.isin(members_in_edges)
]

# Mapeo member_id -> índice entero consecutivo (requerido por PyG)
member_ids = member_features_M.index.tolist()
member_id_to_idx = {mid: i for i, mid in enumerate(member_ids)}

# Filtramos aristas a los nodos presentes
m_edges = member_edges[
    member_edges['member1'].isin(member_id_to_idx) &
    member_edges['member2'].isin(member_id_to_idx)
].copy()

src = m_edges['member1'].map(member_id_to_idx).values
dst = m_edges['member2'].map(member_id_to_idx).values

# Ambas direcciones para grafo no dirigido
edge_index_M = torch.tensor(
    np.stack([np.concatenate([src, dst]),
              np.concatenate([dst, src])], axis=0),
    dtype=torch.long
)

weights = m_edges['weight'].values
edge_weight_M = torch.tensor(
    np.concatenate([weights, weights]),
    dtype=torch.float
)

x_M = torch.tensor(member_features_M.values, dtype=torch.float)

data_M = Data(
    x=x_M,
    edge_index=edge_index_M,
    edge_weight=edge_weight_M
)
data_M.member_ids = member_ids

print(data_M)
print(f'\nNodos : {data_M.num_nodes}')
print(f'Aristas: {data_M.num_edges}')
print(f'Features por nodo: {data_M.num_node_features}')

Data(x=[11371, 3], edge_index=[2, 2352048], edge_weight=[2352048], member_ids=[11371])

Nodos : 11371
Aristas: 2352048
Features por nodo: 3


## 6. Construcción del objeto PyG — Grafo G

El mismo procedimiento que para M, aplicado al grafo de grupos. `group-edges.csv` contiene 456 grupos con aristas (de los 602 totales) — los 146 grupos restantes no comparten ningún miembro con otros grupos y quedan excluidos por la misma razón que los nodos aislados en M.

Las columnas one-hot de `category_name` son de tipo `bool` en versiones recientes de pandas. PyTorch no acepta arrays de tipo `object` ni `bool` directamente, por lo que es necesario convertir explícitamente el DataFrame a `float32` antes de crear el tensor.

In [10]:
# Solo grupos que aparecen en group_edges — descartamos nodos aislados
groups_in_edges = set(group_edges['group1']).union(set(group_edges['group2']))
group_features_G = group_features_scaled[
    group_features_scaled.index.isin(groups_in_edges)
].astype(np.float32)  # conversión necesaria: pd.get_dummies genera bool en pandas recientes

# Mapeo group_id -> índice entero consecutivo
group_ids = group_features_G.index.tolist()
group_id_to_idx = {gid: i for i, gid in enumerate(group_ids)}

g_edges = group_edges[
    group_edges['group1'].isin(group_id_to_idx) &
    group_edges['group2'].isin(group_id_to_idx)
].copy()

src = g_edges['group1'].map(group_id_to_idx).values
dst = g_edges['group2'].map(group_id_to_idx).values

edge_index_G = torch.tensor(
    np.stack([np.concatenate([src, dst]),
              np.concatenate([dst, src])], axis=0),
    dtype=torch.long
)

weights = g_edges['weight'].values
edge_weight_G = torch.tensor(
    np.concatenate([weights, weights]),
    dtype=torch.float
)

x_G = torch.tensor(group_features_G.values, dtype=torch.float)

data_G = Data(
    x=x_G,
    edge_index=edge_index_G,
    edge_weight=edge_weight_G
)
data_G.group_ids = group_ids

print(data_G)
print(f'\nNodos : {data_G.num_nodes}')
print(f'Aristas: {data_G.num_edges}')
print(f'Features por nodo: {data_G.num_node_features}')

Data(x=[456, 35], edge_index=[2, 13384], edge_weight=[13384], group_ids=[456])

Nodos : 456
Aristas: 13384
Features por nodo: 35


## 7. Construcción del objeto PyG — Grafo MG (bipartito)

El grafo bipartito presenta un reto adicional: contiene dos tipos de nodos heterogéneos (miembros y grupos) con conjuntos de features completamente distintos. Mientras que un nodo miembro tiene 3 features y un nodo grupo tiene 35, el objeto `Data` de PyG requiere una única matriz `X` con la misma dimensión para todos los nodos.

La solución estándar en la literatura es construir una **matriz de features unificada** de dimensión `(num_miembros + num_grupos) × (3 + 35) = (25.233 × 38)`, donde:
- Los nodos miembro tienen sus 3 features en las primeras columnas y **ceros** en las 35 restantes
- Los nodos grupo tienen **ceros** en las primeras 3 columnas y sus 35 features en las restantes

Este padding con ceros introduce cierta artificialidad, pero es la aproximación habitual cuando no se dispone de una arquitectura heterogénea (HAN, HGT...) que trate explícitamente los dos tipos de nodos. La alternativa sería usar un modelo heterogéneo de PyG, lo que está fuera del scope de este trabajo.

**Nota sobre nodos del bipartito:** a diferencia de M y G, en el bipartito **no se filtran nodos aislados**, porque por construcción todos los nodos en `member-to-group-edges.csv` tienen al menos una arista. Los IDs se prefiján con `member_` y `group_` para evitar colisiones entre los espacios de IDs de miembros y grupos, siguiendo la misma convención del EDA.

In [11]:
# Prefijamos IDs igual que en el EDA para evitar colisiones
member_group_edges_mg = member_group_edges.copy()
member_group_edges_mg['member_id'] = 'member_' + member_group_edges_mg['member_id'].astype(str)
member_group_edges_mg['group_id']  = 'group_'  + member_group_edges_mg['group_id'].astype(str)

# Nodos únicos en el bipartito
mg_member_ids = sorted(member_group_edges_mg['member_id'].unique())
mg_group_ids  = sorted(member_group_edges_mg['group_id'].unique())

# Índices: miembros primero [0, n_members), grupos después [n_members, n_members+n_groups)
all_nodes = mg_member_ids + mg_group_ids
node_to_idx = {n: i for i, n in enumerate(all_nodes)}

n_members = len(mg_member_ids)
n_groups  = len(mg_group_ids)
n_nodes   = n_members + n_groups

print(f'Nodos miembro en MG: {n_members}')
print(f'Nodos grupo en MG  : {n_groups}')
print(f'Total nodos        : {n_nodes}')

Nodos miembro en MG: 24631
Nodos grupo en MG  : 602
Total nodos        : 25233


In [12]:
n_member_feats = member_features_scaled.shape[1]  # 3
n_group_feats  = group_features_scaled.shape[1]   # 35
n_total_feats  = n_member_feats + n_group_feats    # 38

# Matriz X inicializada a ceros — se rellena por bloques
x_MG = np.zeros((n_nodes, n_total_feats), dtype=np.float32)

# Nodos miembro: features en columnas 0:3, ceros en 3:38
for i, mid in enumerate(mg_member_ids):
    raw_id = int(mid.replace('member_', ''))
    if raw_id in member_features_scaled.index:
        x_MG[i, :n_member_feats] = member_features_scaled.loc[raw_id].values

# Nodos grupo: ceros en columnas 0:3, features en 3:38
for j, gid in enumerate(mg_group_ids):
    raw_id = int(gid.replace('group_', ''))
    if raw_id in group_features_scaled.index:
        x_MG[n_members + j, n_member_feats:] = group_features_scaled.loc[raw_id].values

print(f'Shape matriz X_MG: {x_MG.shape}')
print(f'Nulos: {np.isnan(x_MG).sum()}')

Shape matriz X_MG: (25233, 38)
Nulos: 0


In [13]:
src = member_group_edges_mg['member_id'].map(node_to_idx).values
dst = member_group_edges_mg['group_id'].map(node_to_idx).values

edge_index_MG = torch.tensor(
    np.stack([np.concatenate([src, dst]),
              np.concatenate([dst, src])], axis=0),
    dtype=torch.long
)

weights = member_group_edges_mg['weight'].values
edge_weight_MG = torch.tensor(
    np.concatenate([weights, weights]),
    dtype=torch.float
)

data_MG = Data(
    x=torch.tensor(x_MG, dtype=torch.float),
    edge_index=edge_index_MG,
    edge_weight=edge_weight_MG
)
data_MG.node_ids  = all_nodes
data_MG.n_members = n_members
data_MG.n_groups  = n_groups

print(data_MG)
print(f'\nNodos : {data_MG.num_nodes}')
print(f'Aristas: {data_MG.num_edges}')
print(f'Features por nodo: {data_MG.num_node_features}')

Data(x=[25233, 38], edge_index=[2, 91166], edge_weight=[91166], node_ids=[25233], n_members=24631, n_groups=602)

Nodos : 25233
Aristas: 91166
Features por nodo: 38


## 8. Resumen y guardado

Los tres objetos `Data` se guardan en formato `.pt` (formato nativo de PyTorch) en `../data/graph_data/`. Este formato serializa el objeto completo — incluyendo `x`, `edge_index`, `edge_weight` y los atributos adicionales de mapeo de IDs — y permite cargarlo directamente en los notebooks del GAE con `torch.load()`.

In [14]:
print('=' * 50)
print('RESUMEN DE LOS TRES GRAFOS')
print('=' * 50)
for name, data in [('M', data_M), ('G', data_G), ('MG', data_MG)]:
    print(f'\nGrafo {name}:')
    print(f'  Nodos          : {data.num_nodes}')
    print(f'  Aristas        : {data.num_edges}')
    print(f'  Features/nodo  : {data.num_node_features}')
    print(f'  Shape X        : {data.x.shape}')

RESUMEN DE LOS TRES GRAFOS

Grafo M:
  Nodos          : 11371
  Aristas        : 2352048
  Features/nodo  : 3
  Shape X        : torch.Size([11371, 3])

Grafo G:
  Nodos          : 456
  Aristas        : 13384
  Features/nodo  : 35
  Shape X        : torch.Size([456, 35])

Grafo MG:
  Nodos          : 25233
  Aristas        : 91166
  Features/nodo  : 38
  Shape X        : torch.Size([25233, 38])


In [15]:
torch.save(data_M,  os.path.join(OUTPUT_PATH, 'data_M.pt'))
torch.save(data_G,  os.path.join(OUTPUT_PATH, 'data_G.pt'))
torch.save(data_MG, os.path.join(OUTPUT_PATH, 'data_MG.pt'))

print('Guardado:')
print(f'  {OUTPUT_PATH}data_M.pt')
print(f'  {OUTPUT_PATH}data_G.pt')
print(f'  {OUTPUT_PATH}data_MG.pt')

Guardado:
  ../data/graph_data/data_M.pt
  ../data/graph_data/data_G.pt
  ../data/graph_data/data_MG.pt
